#Goals of this Notebook:
1) Data Analysis
2) Train a Decision Tree Classifier to predict Mortality

#Part 1. Data Analysis

In [22]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
import matplotlib.pyplot as plt

df = pd.read_csv('/content/drive/MyDrive/NYC_Motor_Vehicle_Collisions_to_Person.csv')



In [23]:
def is_valid_test(expected_counts):
  num_gte_five = 0
  num_cells = 0
  for row in expected_counts:
    for cell in row:
      if cell >= 5:
        num_gte_five += 1
      num_cells += 1
  print(f"Chi-Squared test validity: {num_gte_five/num_cells > 0.8}, Value: {num_gte_five/num_cells}")
def chi_square_test (col1, col2):
  contingency_table = pd.crosstab(col1, col2)
  chi2_statistic, p_value, dof, expected_frequencies = chi2_contingency(contingency_table)

  cramers_v = association(contingency_table, method="cramer")
  is_valid_test(expected_frequencies)
  print(f"Chi-square statistic: {chi2_statistic}")
  print(f"P-Value: {p_value}")
  print(f"Degrees of Freedom: {dof}")
  print(f"Expected Frequencies: {expected_frequencies}")
  print(f"Cramer\'s V test for effect size {cramers_v}")

In [ ]:
#Chi-squared test for independence between the variables: Type of Bodily Injury, Mortality
chi_square_test(df["PERSON_INJURY"], df["BODILY_INJURY"])



Code below condenses categories to raise expected frequencies table values to > 5. This makes the chi-square test for independence viable.  


In [ ]:
df['SAFETY_EQUIPMENT'] = df['SAFETY_EQUIPMENT'].apply(
    lambda x:
    'Seat Belt or Harness' if x in [
        "Lap Belt & Harness",
        "Lap Belt",
        "Harness"
    ]

    else 'Seat Belt and Airbag' if x in [
        "Air Bag Deployed/Lap Belt/Harness",
        "Air Bag Deployed/Lap Belt"
    ]

    else 'Airbag Only' if x in [
        "Air Bag Deployed"
    ]

    else 'Child Restraint' if x in [
        "Child Restraint Only",
        "Air Bag Deployed/Child Restraint"
    ]

    else 'Helmet' if x in [
        "Helmet (Motorcycle Only)",
        "Helmet Only (In-Line Skater/Bicyclist)",
        "Helmet/Other (In-Line Skater/Bicyclist)"
    ]


    else 'Other Protective Gear' if x in [
        "Pads Only (In-Line Skater/Bicyclist)",
        "Stoppers Only (In-Line Skater/Bicyclist)"
    ]

    else 'Other/Unknown' if x in [
        "Other",
        "Unknown"
    ]

    else 'Other/Unknown'
)

In [ ]:
#Chi-squared test for independence between the variables: Safety Equipment, Mortality
print(df["SAFETY_EQUIPMENT"].value_counts())
chi_square_test(df["BODILY_INJURY"], df["SAFETY_EQUIPMENT"])

In [ ]:
#ttest for independence between the variables: Age, Mortality
injured = df[(df['PERSON_INJURY'] == 'Injured')
             & (df['PERSON_AGE'] < 130)
             & (df['PERSON_AGE'] > 0)]['PERSON_AGE'].dropna()
dead = df[(df['PERSON_INJURY'] == 'Killed')
            & (df['PERSON_AGE'] < 130)
            & (df['PERSON_AGE'] > 0)]['PERSON_AGE'].dropna()

print(injured.mean(), injured.std())
print(dead.mean(), dead.std(), '\n')
t_stat, p_val = stats.ttest_ind(injured, dead, equal_var=False, alternative='less')
print(t_stat, p_val)

In [ ]:
x = [injured, dead]
plt.boxplot(x, orientation = 'vertical')

Part 2. Classification

Below I train a Decision tree classifier to predict mortality. This model with optimal hyperparameters averages CV performance of about 70% recall. The goal is to build a tree which can be used as __a flowchart__ for making decisions about the severity of a driver's condition after a crash. More serious crashes, which could lead to the driver's death should be handled by healthcare professions as an emergency. Recall score is optimized for since it is imperative to reduce type II errors in this model as misclassifications of drivers as 'injured' would understate their issue and potentially lead them to not recieve adequate care.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn import tree
from sklearn.tree import export_text
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


df = pd.read_csv("/content/drive/MyDrive/NYC_Motor_Vehicle_Collisions_to_Person.csv")

X = df.drop('PERSON_INJURY', axis=1)
y = df['PERSON_INJURY'].apply(lambda x: 1 if x == 'Killed' else 0)


print(f"Value counts: {y.value_counts()}")


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)


#Cleaning '?': Imputation with mode, 'np.nan': Imputation with median
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
   X_train[col] = X_train[col].replace("?", X_train[col].mode()[0])
   X_test[col] = X_test[col].replace("?", X_train[col].mode()[0])
numerical_cols = X_train.select_dtypes(exclude=['object']).columns.tolist()
for col in numerical_cols:
   X_train[col] = X_train[col].fillna(X_train[col].median())
   X_test[col] = X_test[col].fillna(X_train[col].median())


#One hot encoding
preprocessor = ColumnTransformer(
   transformers=[
       ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
   ],
   remainder="passthrough"
)
pipeline = Pipeline(steps=[
   ("preprocessor", preprocessor),
   ("clf", tree.DecisionTreeClassifier())
])
####


#Training


param_grid = {
   'clf__max_depth': [1,2,3,4],
}
rs = RandomizedSearchCV(estimator=pipeline, param_distributions=param_grid,
n_iter=5, cv=10, verbose=2, n_jobs=-1, scoring="recall")


rs.fit(X_train, y_train)


results_df = pd.DataFrame(rs.cv_results_)
print("Best Params:", rs.best_params_)
print(results_df)
######


#Final Test & eval
y_pred = rs.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)


print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

#Display results

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels
                             =["Injured", "Killed"])
disp.plot(cmap=plt.cm.Greens)
plt.title("Random Forests Confusion Matrix (Test Set)")
plt.show()